### 文件加载效果测试

In [ ]:
from rag.loaders import RapidOCRImageLoader
loader = RapidOCRImageLoader()
loader.load()

In [ ]:
from rag.loaders import LLMCaptionImageLoader
loader = LLMCaptionImageLoader()
loader.load()

In [ ]:
from rag.loaders import BLIPCaptionImageLoader
loader = BLIPCaptionImageLoader()
loader.load()

In [ ]:
from rag.loaders.pdf import PDFLoader
loader = PDFLoader()
loader.load()

In [ ]:
from rag.loaders.doc import DOCLoader
loader = DOCLoader()
loader.load()

In [ ]:
from rag.loaders.ppt import PPTLoader
loader = PPTLoader()
loader.load()

### 文本分割效果测试

In [ ]:
from rag.loaders.pdf import PDFLoader
loader = PDFLoader("../test.pdf")
doc = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    keep_separator="end", is_separator_regex=True, 
    chunk_size=chunk_size, chunk_overlap=chunk_overlap,
    separators=[r'[。！？]+\s*', r'[；，]+\s*', r'[.!?]+\s*', r'[;,]+\s*', r'\s+'])
docs = text_splitter.split_documents(doc)
docs

### 向量数据库测试

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-0.6b",
    base_url="http://192.168.12.242:8780/v1",
    api_key="gpustack_a2cef91c28b3eba7_373b72b1287951f7945afa9a844448e5"
)

from langchain_core.documents import Document
texts = [
    "人工智能是一种模拟人类智能的技术。",
    "大语言模型是人工智能的一个重要分支。",
    "LangChain 是用于开发 LLM 应用的框架。",
    "FAISS 是 Facebook 开发的高效向量搜索库。",
    "RAG 是 Retrieval-Augmented Generation 的缩写。",
]
documents = [Document(page_content=text) for text in texts]

In [ ]:
from rag.utils import get_vectorstore
vectorstore = get_vectorstore(
    kb_name="test",
    kb_type="Faiss",
    embedding=embeddings,
    vectorstores_path="./data/kbs/test/vectorstores",
)
vectorstore.add_documents(documents)

In [ ]:
from rag.utils import get_vectorstore
vectorstore2 = get_vectorstore(
    kb_name="example",
    kb_type="Faiss",
    embedding=embeddings,
    vectorstores_path="./data/kbs/test/vectorstores",
)
vectorstore2.similarity_search("faiss", k=2)

### 信息数据库测试

In [ ]:
from kbm.kbs_table import add_kb_to_db, get_kb_from_db, delete_kb_from_db, get_kb_names
# add_kb_to_db("test","Milvus","这是一个测试","Xinference","123","321","bge")
# get_kb_from_db("test")
# delete_kb_from_db("test")
get_kb_names()

In [ ]:
from kbm.files_table import add_files_to_db, get_files_from_db, delete_files_from_db
# add_files_to_db(kb_id=1, file_infos=[
    # {"file_name": "test.txt", "file_size": 1024, "file_type": "txt", "file_dir": "test/", "loader_name": "test", "splitter_name": "test"},
    # {"file_name": "test1.pdf", "file_size": 1024, "file_type": "pdf", "file_dir": "test/", "loader_name": "test", "splitter_name": "test"},
    # {"file_name": "test2.docx", "file_size": 1024, "file_type": "docx", "file_dir": "test/", "loader_name": "test", "splitter_name": "test"},
# ])
get_files_from_db(kb_id=1)
# delete_files_from_db(kb_id=1, file_names=["test1.pdf", "test2.docx"])

In [ ]:
from kbm.chunks_table import add_chunks_to_db, get_chunks_from_db, delete_chunks_from_db, get_vector_ids
# add_chunks_to_db(kb_id=1, file_id=1, vector_ids=[1, 2, 3, 11, 12, 13, 111, 123, 134, 1111, 1234, 1345, 11111, 12345])
# get_chunks_from_db(kb_id=1, file_id=1)
# delete_chunks_from_db(kb_id=1, file_id=1, vector_ids=[1, 2, 3])
get_vector_ids(kb_id=1, file_id=1)

### LangGraph

#### 简单对话

In [ ]:
import openai
from langchain_openai import ChatOpenAI
from langchain_core.outputs import ChatResult, ChatGenerationChunk

class ChatOpenAIWithReasoning(ChatOpenAI):
    """支持 reasoning_content 的 ChatOpenAI"""

    def _create_chat_result(
        self,
        response: dict | openai.BaseModel,
        generation_info: dict | None = None,
    ) -> ChatResult:
        """提取结果中的 reasoning_content 到 additional_kwargs"""
        chat_result = super()._create_chat_result(
            response, generation_info
        )
        if (isinstance(response, openai.BaseModel)
            and (choices := getattr(response, "choices", None))):
            reasoning_content = (
                getattr(choices[0].message, "reasoning", None) or
                getattr(choices[0].message, "reasoning_content", None)
            )
            if reasoning_content is not None:
                chat_result.generations[0].message.additional_kwargs["reasoning_content"] = reasoning_content
        return chat_result
    
    def _convert_chunk_to_generation_chunk(
        self,
        chunk: dict,
        default_chunk_class: type,
        base_generation_info: dict | None,
    ) -> ChatGenerationChunk | None:
        """提取流式响应中的 reasoning_content 到 additional_kwargs"""
        generation_chunk = super()._convert_chunk_to_generation_chunk(
            chunk, default_chunk_class, base_generation_info
        )
        if (generation_chunk
            and (choices := chunk.get("choices", [])) 
            and (delta := choices[0].get("delta", {}))):
            reasoning_content = (
                delta.get("reasoning") or
                delta.get("reasoning_content")
            )
            if reasoning_content is not None:
                generation_chunk.message.additional_kwargs["reasoning_content"] = reasoning_content
        return generation_chunk

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_qwq.chat_models import ChatQwen
from langchain_deepseek import ChatDeepSeek
from langchain_litellm import ChatLiteLLM
# 1. 初始化模型对象
llm = ChatOpenAIWithReasoning(
    # model="qwen3.5-122b", 
    # base_url="http://192.168.12.242:8780/v1", 
    # api_key="gpustack_a2cef91c28b3eba7_373b72b1287951f7945afa9a844448e5",
    model="glm-4.5-flash",
    base_url="https://open.bigmodel.cn/api/paas/v4",
    api_key="2d0e3a459c3543f9a9b630f25fae4706.irY241kbUc6BZGyd",
    temperature=0.7,
    extra_body={
        "thinking": {
            "type": "enabled",
            # "budget_tokens": thinking
        },
        "chat_template_kwargs": {
            "enable_thinking": True,
            "return_reasoning": True
        }
    },
    streaming=True,   # 可以启用流式
    stream_usage=True # 可选，获取 token 使用统计
)

# 2. 调用模型
# response = llm.invoke("你好")

# 3. 提取内容
# 注意：通过 langchain_openai 调用时，推理内容的位置可能因服务实现而异。
# 如果服务端支持且正确返回，它可能会出现在 additional_kwargs 中。
for event in llm.stream("你好"):
    print(event)
# print(response)
# print("最终答案:", response.content)
# print("额外信息 (可能包含推理内容):", response.additional_kwargs)
# 你可以检查 response.additional_kwargs 中是否包含 'reasoning_content' 字段

In [ ]:
import streamlit
streamlit.status()
a=streamlit.empty()
a._markdown
a.markdown

In [16]:
a={}

In [17]:
if b := a.get("a"):
    print(b)
else:
    print(0)

0


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

def create_graph(llm):
    graph = StateGraph(MessagesState)
    def chatbot(state: MessagesState):
        return {"messages": [llm.invoke(state["messages"])]}
    graph.add_node("chabot", chatbot)
    graph.add_edge(START, "chabot")
    graph.add_edge("chabot", END)
    return graph.compile()

In [ ]:
from PIL import Image
import io
a=graph.get_graph().draw_mermaid_png()
image = Image.open(io.BytesIO(a))
image

In [ ]:
graph = create_graph(llm=llm)
def graph_response(user_input: str):
    for event in graph.stream(
        input={"messages": [{"role": "user", "content": user_input}]},
        # config={"configurable": {"thread_id": "1"}},
        stream_mode="messages"
        ):
        print(event)
        # print(event[0].content, end='', flush=True)
        # for value in event.values():
            # print("Assistant:", value["messages"][-1].content)
# graph_response("写一首七言律诗表达对对未来科技的遐想")
graph_response("写一首词（沁园春）表达对对未来科技的遐想")

#### 工具调用

In [ ]:
import requests
from langchain_core.tools import tool
LANGSEARCH_API_KEY = "sk-77c31f90bb574507ab86a2875654d027"

@tool
def langsearch_websearch(query: str, count: int = 10, freshness: str = "noLimit"):
    """
    使用LangSearch Web Search API执行网络搜索。

    参数：
    - query：搜索关键词
    - count：返回的搜索结果数量
    - freshness：搜索结果的新鲜度，可选值为"noLimit", "oneDay", "oneWeek", "oneMonth", "oneYear"。

    返回值：
    - 搜索结果的详细信息，包括网页标题、网页URL、网页内容、网页发布时间等。
    """
    url = "https://api.langsearch.com/v1/web-search"
    headers = {
        "Authorization": f"Bearer {LANGSEARCH_API_KEY}",
        "Content-Type": "application/json",
    }
    json = {
        "query": query,
        "freshness": freshness,  # "oneDay", "oneWeek", "oneMonth", "oneYear", "noLimit"
        "summary": True,
        "count": count,
    }
    response = requests.post(url, headers=headers, json=json)

    if response.status_code == 200:
        json_response = response.json()
        try:
            if json_response["code"] != 200 or not json_response["data"]:
                return f"Search API request failed, reason: {response.msg or 'Unknown error'}"
            
            webpages = json_response["data"]["webPages"]["value"]
            if not webpages:
                return "No relevant results found."
            formatted_results = ""
            for idx, page in enumerate(webpages, start=1):
                formatted_results += (
                    f"Citation: {idx}\n"
                    f"Title: {page['name']}\n"
                    f"URL: {page['url']}\n"
                    f"Content: {page['summary']}\n"
                )
            return formatted_results.strip()
        except Exception as e:
            return f"Search API request failed, reason: Failed to parse search results {str(e)}"
    else:
        return f"Search API request failed, status code: {response.status_code}, error message: {response.text}"

tools = [langsearch_websearch]
langsearch_websearch.invoke("今天宁波奉化的天气")

In [ ]:
from langchain_core.tools import tool
@tool
def query_db(input:str):
    """查询数据库信息

    args:
        input: 输入问题
    """
    import time
    time.sleep(30)
    return "查询结果：你叫小zhi"
@tool
def multiply(a: int, b: int) -> int:
    """计算 a 乘 b
    Args:
        a: 乘数
        b: 乘数
    """
    import time
    time.sleep(20)
    return a * b
@tool
def add(a: int, b: int) -> int:
    """计算 a 加 b"""
    return a + b
tools =[query_db, multiply, add]

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition

def create_graph(llm, tools):
    graph = StateGraph(MessagesState)
    llm_with_tools = llm.bind_tools(tools)
    def chat_node(state: MessagesState):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}
    tool_node = ToolNode(tools=tools)
    graph.add_node("chatbot", chat_node)
    graph.add_node("tools", tool_node)
    graph.add_edge(START, "chatbot")
    graph.add_conditional_edges("chatbot", tools_condition)
    graph.add_edge("tools", "chatbot")
    return graph.compile(checkpointer=MemorySaver())
graph = create_graph(llm=llm, tools=tools)

In [ ]:
from PIL import Image
import io
a=graph.get_graph().draw_mermaid_png()
image = Image.open(io.BytesIO(a))
image

In [ ]:
from langchain_core.messages import 

In [ ]:
from langchain_core.messages import AIMessageChunk, ToolMessage
def graph_response(user_input: str):
    for event in graph.stream(
        input={"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"thread_id": 1}},
        stream_mode="messages",
    ):
        print(event)
        # if type(event[0]) == AIMessageChunk:
        #     if len(event[0].tool_calls):
        #         print("正在调用工具 . . .", event[0].tool_calls[0]["name"], event[0].tool_calls[0]["args"])
        #     else:
        #         print(event[0].content, end="", flush=True)
        # elif type(event[0]) == ToolMessage:
        #     print("工具调用结果：", event[0].content)
        # event["messages"][-1].pretty_print()
        # for value in event.values():
            # print("Assistant:", value["messages"][-1].content)
# graph_response("今天宁波奉化的天气")
# graph_response("今天和AI有关的热搜")
# graph_response("搜索一周内的十大AI热点")
# graph_response("写一首词（沁园春）表达对今日热搜的思考")
graph_response("计算1+2x3")

#### END

In [ ]:
tool_calls=[
    {
        'name': 'add',
        'args': {'a': 1234, 'b': 2345},
        'id': 'call_a6f46abc9c974b1987a9f570',
        'type': 'tool_call'
    },
    {
        'name': 'multiply',
        'args': {'a': 2, 'b': 2},
        'id': 'call_c077cca92497475d87f02744',
        'type': 'tool_call'
    }
]

a="正在调用 " + " ".join([f"`{tc['name']}`" for tc in tool_calls])
a

In [ ]:
import streamlit as st
a=st.status("a")
a.c